# Refatoração em PySpark — Escalabilidade da Camada Gold

> **Pessoa 4 — Etapa 1.** Demonstra escalabilidade reimplementando o *join* estrutural
> Silver→Gold e agregações em **PySpark**, e compara o desempenho com **Pandas**.
>
> Tudo aqui é processado no motor distribuído do Spark (lado JVM): leitura/escrita Parquet,
> `join`, `groupBy`+`agg` e *window function*. **Não** usamos `createDataFrame` a partir de
> dados Python nem UDFs (incompatíveis com Python 3.13 + PySpark no Windows).

## Visão geral

```mermaid
graph TD
    A[(Silver Parquets)] --> B[Etapa A: read + join LEFT por incident_id]
    B --> C[(data/gold/spark_join.parquet)]
    C --> D[Etapa B: groupBy + agg]
    C --> E[Etapa B: window row_number top-5/ano]
    D --> F[(data/gold/spark_agg_by_vector.parquet)]
    E --> G[(data/gold/spark_top5_per_year.parquet)]
    A --> H[Benchmark Pandas vs Spark]
```


In [ ]:
# Setup do Spark local e caminhos do projeto
import os
from pathlib import Path
import time
import pandas as pd

# Garante que driver/workers usem o mesmo Python (o do venv que executa este notebook)
import sys
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

PROJECT_ROOT = Path("..").resolve()
SILVER = PROJECT_ROOT / "data" / "silver"
GOLD = PROJECT_ROOT / "data" / "gold"
GOLD.mkdir(parents=True, exist_ok=True)

spark = (SparkSession.builder
         .appName("CyberGoldRefactor")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.driver.memory", "4g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("SILVER:", SILVER)
print("GOLD:", GOLD)

## Etapa A — Leitura e Join Silver → Gold (estrutural)

Equivalente PySpark do *join* da Pessoa 2, na versão **apenas estrutural** (sem encoding/scaling):
`incidents LEFT JOIN financial LEFT JOIN market` pela chave `incident_id`.

> **Nota de schema:** a coluna `stock_ticker` existe **tanto em `incidents` quanto em `market`**.
> Para evitar uma coluna duplicada/ambígua no resultado (que faria o `write.parquet` falhar),
> descartamos o `stock_ticker` de `market` antes do join e mantemos o de `incidents`.

**Requisitos cobertos:** leitura Parquet ✓ · join ✓ · escrita Parquet ✓.

In [ ]:
inc = spark.read.parquet((SILVER / "incidents_master_silver.parquet").as_posix())
fin = spark.read.parquet((SILVER / "financial_impact_silver.parquet").as_posix())
mkt = spark.read.parquet((SILVER / "market_impact_silver.parquet").as_posix())

print(f"incidents: {inc.count()} | financial: {fin.count()} | market: {mkt.count()}")

# Colisao de nome: 'stock_ticker' em incidents e em market -> manter o de incidents
mkt = mkt.drop("stock_ticker")

joined = (inc
          .join(fin, on="incident_id", how="left")
          .join(mkt, on="incident_id", how="left"))

# Relatorio de match do join
n_total = joined.count()
n_fin = joined.filter(F.col("total_loss_usd").isNotNull()).count()
n_mkt = joined.filter(F.col("market_cap_at_disclosure").isNotNull()).count()
print(f"Join: {n_total} linhas x {len(joined.columns)} colunas")
print(f"  - com match em financial: {n_fin}")
print(f"  - com match em market:    {n_mkt}")

out_join = (GOLD / "spark_join.parquet").as_posix()
joined.write.mode("overwrite").parquet(out_join)
print("[OK] escrito:", out_join)

## Etapa B — Agregações e Window Function

Duas operações analíticas sobre o resultado do join:

1. **`groupBy` + agregação** — perda por vetor de ataque (contagem, média e mediana de `total_loss_usd`).
2. **Window function** — ranking dos 5 incidentes de maior perda por ano (`row_number`).

**Requisitos cobertos:** groupBy + agg ✓ · window function ✓ · escrita Parquet ✓.

In [ ]:
df = spark.read.parquet((GOLD / "spark_join.parquet").as_posix())

# groupBy + agregacao: perda por vetor de ataque
agg_by_vector = (df.groupBy("attack_vector_primary")
                   .agg(F.count("*").alias("n_incidents"),
                        F.round(F.avg("total_loss_usd"), 2).alias("avg_loss_usd"),
                        F.expr("percentile_approx(total_loss_usd, 0.5)").alias("median_loss_usd"))
                   .orderBy(F.desc("n_incidents")))
agg_by_vector.show(truncate=False)

agg_by_vector.write.mode("overwrite").parquet((GOLD / "spark_agg_by_vector.parquet").as_posix())
print("[OK] escrito: spark_agg_by_vector.parquet")

In [ ]:
# Window function: top-5 incidentes por perda, por ano
w = Window.partitionBy("incident_year").orderBy(F.desc("total_loss_usd"))
ranked = (df.withColumn("rank_loss_year", F.row_number().over(w))
            .filter(F.col("rank_loss_year") <= 5)
            .select("incident_year", "incident_id", "attack_vector_primary",
                    "total_loss_usd", "rank_loss_year")
            .orderBy("incident_year", "rank_loss_year"))
ranked.show(20, truncate=False)

ranked.write.mode("overwrite").parquet((GOLD / "spark_top5_per_year.parquet").as_posix())
print("[OK] escrito: spark_top5_per_year.parquet")

## Benchmark — Pandas vs PySpark

Comparamos a **mesma Etapa A (read + join)** nas duas tecnologias, medindo o tempo de parede
com `time.perf_counter()`. O `stock_ticker` duplicado é tratado igual nos dois lados para a
comparação ser justa.

In [ ]:
# --- Pandas ---
t0 = time.perf_counter()
inc_p = pd.read_parquet(SILVER / "incidents_master_silver.parquet")
fin_p = pd.read_parquet(SILVER / "financial_impact_silver.parquet")
mkt_p = pd.read_parquet(SILVER / "market_impact_silver.parquet").drop(columns=["stock_ticker"])
joined_p = (inc_p
            .merge(fin_p, on="incident_id", how="left")
            .merge(mkt_p, on="incident_id", how="left"))
joined_p.to_parquet(GOLD / "pandas_join.parquet", index=False)
pandas_time = time.perf_counter() - t0
print(f"Pandas: join {joined_p.shape} em {pandas_time:.3f}s")

In [ ]:
# --- PySpark (mesma Etapa A, medindo do read ao write) ---
t0 = time.perf_counter()
inc_s = spark.read.parquet((SILVER / "incidents_master_silver.parquet").as_posix())
fin_s = spark.read.parquet((SILVER / "financial_impact_silver.parquet").as_posix())
mkt_s = spark.read.parquet((SILVER / "market_impact_silver.parquet").as_posix()).drop("stock_ticker")
joined_s = (inc_s.join(fin_s, on="incident_id", how="left")
                 .join(mkt_s, on="incident_id", how="left"))
joined_s.write.mode("overwrite").parquet((GOLD / "spark_join_bench.parquet").as_posix())
spark_count = joined_s.count()
spark_time = time.perf_counter() - t0
print(f"Spark: join ({spark_count}, {len(joined_s.columns)}) em {spark_time:.3f}s")

In [ ]:
# Tabela comparativa
bench = pd.DataFrame([
    {"tecnologia": "Pandas",  "tempo_s": round(pandas_time, 3),
     "overhead_init": "~0s",        "memoria": "RAM da maquina",
     "vence_quando": "volume pequeno (< 1M linhas)"},
    {"tecnologia": "PySpark", "tempo_s": round(spark_time, 3),
     "overhead_init": "~3-5s (JVM)", "memoria": "particionavel/distribuida",
     "vence_quando": "volume grande (> 10M linhas, clusters)"},
])
print("=== BENCHMARK PANDAS vs PYSPARK ===")
print(bench.to_string(index=False))
print(f"\nNeste volume (~{joined_p.shape[0]} linhas), Spark foi {spark_time/pandas_time:.1f}x mais lento que Pandas.")

### Discussão do benchmark

| Critério | Pandas | PySpark |
|----------|--------|---------|
| Tempo (volume atual, ~850 linhas) | menor | maior |
| Overhead de inicialização | ~0s | ~3–5s (subida da JVM + sessão) |
| Limite de memória | RAM da máquina (single-node) | particionável/distribuível |
| Onde vence | datasets pequenos (< ~1M linhas) | datasets grandes (> 10M) e ambientes distribuídos |

**Conclusão honesta:** neste volume o **Pandas vence**, porque o custo fixo do Spark (subir a JVM,
serialização, agendamento de tarefas) domina o tempo total quando há poucos dados. O ganho real do
PySpark aparece em **escala** — dados que não cabem na RAM de uma máquina, ou processamento
paralelo em cluster (Databricks, EMR, etc.). O objetivo desta etapa é **demonstrar a refatoração
para uma API distribuída**, não vencer um benchmark de brinquedo: o mesmo código roda sem alteração
sobre volumes ordens de magnitude maiores.

In [ ]:
spark.stop()
print("SparkSession encerrada.")